In [1]:
#STEP 1: Load libraries
import pandas as pd
import numpy as np
import shap

from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
from sklearn.utils.class_weight import compute_sample_weight

import joblib

In [2]:
#STEP 2: Load and Check data

# Load the ClinicalTrials.gov CSV file
df = pd.read_csv("ctg-studies.csv")

print("Rows and columns:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nStudy Status counts:")
print(df["Study Status"].value_counts())

print("\nMissing values percentage:")
print((df.isnull().mean().sort_values(ascending=False) * 100).head(10))

df.head()

Rows and columns: (61215, 18)

Columns:
['NCT Number', 'Study Status', 'Brief Summary', 'Conditions', 'Interventions', 'Sponsor', 'Collaborators', 'Sex', 'Age', 'Phases', 'Enrollment', 'Funder Type', 'Study Type', 'Study Design', 'Start Date', 'Primary Completion Date', 'Completion Date', 'Locations']

Study Status counts:
Study Status
COMPLETED     46970
TERMINATED     9115
WITHDRAWN      4670
SUSPENDED       460
Name: count, dtype: int64

Missing values percentage:
Collaborators              69.314710
Locations                   4.946500
Completion Date             0.243404
Sex                         0.026137
Enrollment                  0.011435
Study Design                0.006534
Primary Completion Date     0.000000
Start Date                  0.000000
Study Type                  0.000000
Funder Type                 0.000000
dtype: float64


,NCT Number,Study Status,Brief Summary,Conditions,Interventions,Sponsor,Collaborators,Sex,Age,Phases,Enrollment,Funder Type,Study Type,Study Design,Start Date,Primary Completion Date,Completion Date,Locations
0,NCT05185284,COMPLETED,This is open-labe randomized multicenter compa...,COVID-19,DRUG: Favipiravir|DRUG: Favipiravir|DRUG: Remd...,"Promomed, LLC",Solyur Pharmaceuticals Group,ALL,"ADULT, OLDER_ADULT",PHASE3,217.0,OTHER,INTERVENTIONAL,Allocation: RANDOMIZED | Intervention Model: P...,2021-08-11,2021-12-30,2021-12-30,"Regional budgetary health care institution ""Iv..."
1,NCT04752033,COMPLETED,This research study is being done to determine...,"Colorectal Surgery|Pain, Postoperative|Analges...",DRUG: Morphine|DRUG: Hydromorphone,Mayo Clinic,NaN,ALL,"ADULT, OLDER_ADULT",PHASE4,80.0,OTHER,INTERVENTIONAL,Allocation: RANDOMIZED | Intervention Model: S...,2021-03-10,2023-02-17,2023-02-17,"Mayo Clinic in Rochester, Rochester, Minnesota..."
2,NCT06241352,COMPLETED,The results of previous studies conducted by o...,Advanced Pancreatic Cancer,DRUG: statin addition to chemotherapy,Changhai Hospital,NaN,ALL,"ADULT, OLDER_ADULT",PHASE2,42.0,OTHER,INTERVENTIONAL,Allocation: NA | Intervention Model: SINGLE_GR...,2024-03-20,2025-04-30,2025-04-30,"Changhai Hospital, Shanghai, Shanghai Municipa..."
3,NCT03988933,COMPLETED,Rationale:\n\nShorter regimens of high dose da...,Latent Tuberculosis,DRUG: Rifampin double dose|DRUG: Rifampin trip...,McGill University Health Centre/Research Insti...,Canadian Institutes of Health Research (CIHR),ALL,"CHILD, ADULT, OLDER_ADULT",PHASE2,1368.0,OTHER,INTERVENTIONAL,Allocation: RANDOMIZED | Intervention Model: P...,2019-09-20,2023-01-31,2024-12-31,"Unviversity of Calgary, Calgary, Alberta, Cana..."
4,NCT06149247,COMPLETED,The objective of this clinical study is to com...,Cutaneous T-Cell Lymphoma/Mycosis Fungoides,DRUG: Hypericin|DRUG: Mechlorethamine Topical Gel,Soligenix,NaN,ALL,"ADULT, OLDER_ADULT",PHASE2,10.0,INDUSTRY,INTERVENTIONAL,Allocation: RANDOMIZED | Intervention Model: P...,2023-12-05,2024-05-31,2024-06-27,"Rochester Skin Lymphoma Medical Group, Fairpor..."


In [3]:
#STEP 3: Create target 

# Create classification target
# 1 = Completed, 0 = Not completed
df["target_status"] = df["Study Status"].map({
    "COMPLETED": 1,
    "TERMINATED": 0,
    "WITHDRAWN": 0,
    "SUSPENDED": 0
})

print("Target counts:")
print(df["target_status"].value_counts())

print("\nTarget percentage:")
print(df["target_status"].value_counts(normalize=True) * 100)

Target counts:
target_status
1    46970
0    14245
Name: count, dtype: int64

Target percentage:
target_status
1    76.72956
0    23.27044
Name: proportion, dtype: float64


In [4]:
#STEP 4: Create date features

# Convert date columns to datetime format
df["Start Date"] = pd.to_datetime(df["Start Date"], errors="coerce", format="mixed")
df["Completion Date"] = pd.to_datetime(df["Completion Date"], errors="coerce", format="mixed")
df["Primary Completion Date"] = pd.to_datetime(df["Primary Completion Date"], errors="coerce", format="mixed")

# Create date-based features
df["start_year"] = df["Start Date"].dt.year

# Create "duration_months" column
df["duration_months"] = (
    (df["Completion Date"] - df["Start Date"]).dt.days / 30.44
)

df[[
    "Study Status", "target_status", "Start Date",
    "Completion Date", "start_year", "duration_months"
]].head()

print("Missing duration values:", df["duration_months"].isna().sum())
print("Negative duration values:", (df["duration_months"] < 0).sum())

Missing duration values: 149
Negative duration values: 0


In [5]:
#STEP 5: Create predictors(X) and target(y)

# Columns removed because they are leakage, identifiers, or unsuitable for this first model
remove_cols = [
    "NCT Number",
    "Study Status",
    "Start Date",
    "Primary Completion Date",
    "Completion Date",
    "duration_months",
    "target_status",
    "Brief Summary",
    "Collaborators",
    "Study Type"
]

X = df.drop(columns=remove_cols, errors="ignore")
y = df["target_status"]

print("Predictor columns:")
print(X.columns.tolist())

print("\nX shape:", X.shape)
print("y shape:", y.shape)

Predictor columns:
['Conditions', 'Interventions', 'Sponsor', 'Sex', 'Age', 'Phases', 'Enrollment', 'Funder Type', 'Study Design', 'Locations', 'start_year']

X shape: (61215, 11)
y shape: (61215,)


In [6]:
# STEP 6: Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True) * 100)

print("\nTesting target distribution:")
print(y_test.value_counts(normalize=True) * 100)

Training shape: (48972, 11)
Testing shape: (12243, 11)

Training target distribution:
target_status
1    76.72956
0    23.27044
Name: proportion, dtype: float64

Testing target distribution:
target_status
1    76.72956
0    23.27044
Name: proportion, dtype: float64


In [7]:
#STEP 7: Feature engineering 

class FeatureBuilder(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        sponsor = X["Sponsor"].fillna("Unknown")
        self.sponsor_counts_ = sponsor.value_counts()
        return self

    def transform(self, X):
        X = X.copy()
        
        # Fill only the fields needed for feature creation
        for col in ["Sex", "Funder Type", "Study Design"]:
            X[col] = X[col].fillna("Unknown")

        
        X["Conditions"] = X["Conditions"].fillna("")
        X["Interventions"] = X["Interventions"].fillna("")

        # Count features
        X["condition_count"] = X["Conditions"].apply(
            lambda x: len([v for v in str(x).split("|") if v.strip()])
        )

        X["intervention_count"] = X["Interventions"].apply(
            lambda x: len([v for v in str(x).split("|") if v.strip()])
        )

        # Location features
        loc = X["Locations"]
        loc_text = loc.fillna("").str.lower()

        X["has_location"] = loc.notna().astype(int)

        X["location_count"] = loc.fillna("").apply(
            lambda x: len([v for v in str(x).split("|") if v.strip()])
        )

        X["is_multisite"] = (X["location_count"] > 1).astype(int)

        X["has_us_location"] = loc_text.str.contains("united states", regex=False).astype(int)
        X["has_uk_location"] = loc_text.str.contains("united kingdom", regex=False).astype(int)
        X["has_china_location"] = loc_text.str.contains("china", regex=False).astype(int)
        X["has_canada_location"] = loc_text.str.contains("canada", regex=False).astype(int)

        # Age flags using exact matching
        def has_age_group(value, group):
            values = [v.strip() for v in str(value).upper().split(",")]
            return int(group in values)

        X["has_child"] = X["Age"].apply(lambda x: has_age_group(x, "CHILD"))
        X["has_adult"] = X["Age"].apply(lambda x: has_age_group(x, "ADULT"))
        X["has_older_adult"] = X["Age"].apply(lambda x: has_age_group(x, "OLDER_ADULT"))

        # Phase flags using exact matching
        def has_phase(value, phase):
            values = [v.strip() for v in str(value).upper().split("|")]
            return int(phase in values)

        X["has_early_phase1"] = X["Phases"].apply(lambda x: has_phase(x, "EARLY_PHASE1"))
        X["has_phase1"] = X["Phases"].apply(lambda x: has_phase(x, "PHASE1"))
        X["has_phase2"] = X["Phases"].apply(lambda x: has_phase(x, "PHASE2"))
        X["has_phase3"] = X["Phases"].apply(lambda x: has_phase(x, "PHASE3"))
        X["has_phase4"] = X["Phases"].apply(lambda x: has_phase(x, "PHASE4"))

        # Intervention type flags
        intervention = X["Interventions"].fillna("").str.upper()

        X["has_drug"] = intervention.str.contains("DRUG:", regex=False).astype(int)
        X["has_device"] = intervention.str.contains("DEVICE:", regex=False).astype(int)
        X["has_biological"] = intervention.str.contains("BIOLOGICAL:", regex=False).astype(int)
        X["has_procedure"] = intervention.str.contains("PROCEDURE:", regex=False).astype(int)
        X["has_behavioral"] = intervention.str.contains("BEHAVIORAL:", regex=False).astype(int)

        # Covers both possible spellings
        X["has_diagnostic_test"] = (
            intervention.str.contains("DIAGNOSTIC_TEST:", regex=False) |
            intervention.str.contains("DIAGNOSTIC TEST:", regex=False)
        ).astype(int)

        # Sponsor frequency learned from training data
        sponsor = X["Sponsor"].fillna("Unknown")
        X["sponsor_frequency"] = sponsor.map(self.sponsor_counts_).fillna(1)

        # Extract useful parts from Study Design
        def get_design_value(text, key):
            for part in str(text).split("|"):
                part = part.strip()
                if part.upper().startswith(key.upper() + ":"):
                    return part.split(":", 1)[1].strip()
            return "Unknown"

        X["allocation"] = X["Study Design"].apply(lambda x: get_design_value(x, "Allocation"))
        X["intervention_model"] = X["Study Design"].apply(lambda x: get_design_value(x, "Intervention Model"))
        X["masking"] = X["Study Design"].apply(lambda x: get_design_value(x, "Masking"))
        X["primary_purpose"] = X["Study Design"].apply(lambda x: get_design_value(x, "Primary Purpose"))

        return X

builder = FeatureBuilder()

X_train_fe = builder.fit_transform(X_train)
X_test_fe = builder.transform(X_test)

print("Before feature engineering:", X_train.shape, X_test.shape)
print("After feature engineering:", X_train_fe.shape, X_test_fe.shape)

print("Age flags")
display(X_train_fe[["Age", "has_child", "has_adult", "has_older_adult"]].head(5))

print("Phase flags")
display(X_train_fe[["Phases", "has_early_phase1", "has_phase1", "has_phase2", "has_phase3", "has_phase4"]].head(5))

print("Location features")
display(X_train_fe[["Locations", "has_location", "location_count", "is_multisite",
                    "has_us_location", "has_uk_location", "has_china_location", "has_canada_location"]].head(5))

print("Sponsor frequency")
display(X_train_fe[["Sponsor", "sponsor_frequency"]].head(5))

print("Study design extraction")
display(X_train_fe[["Study Design", "allocation", "intervention_model", "masking", "primary_purpose"]].head(5))

print("Condition and intervention features")
display(X_train_fe[["Conditions", "condition_count", "Interventions", "intervention_count",
                    "has_drug", "has_device", "has_biological", "has_procedure",
                    "has_behavioral", "has_diagnostic_test"]].head(5))

Before feature engineering: (48972, 11) (12243, 11)
After feature engineering: (48972, 39) (12243, 39)
Age flags


,Age,has_child,has_adult,has_older_adult
22202,"ADULT, OLDER_ADULT",0,1,1
45439,"ADULT, OLDER_ADULT",0,1,1
44247,"ADULT, OLDER_ADULT",0,1,1
23712,"ADULT, OLDER_ADULT",0,1,1
27905,"ADULT, OLDER_ADULT",0,1,1


Phase flags


,Phases,has_early_phase1,has_phase1,has_phase2,has_phase3,has_phase4
22202,PHASE1,0,1,0,0,0
45439,PHASE4,0,0,0,0,1
44247,PHASE2,0,0,1,0,0
23712,PHASE1,0,1,0,0,0
27905,PHASE2,0,0,1,0,0


Location features


...

,Sponsor,sponsor_frequency
22202,Prelude Therapeutics,9
45439,The Cleveland Clinic,44
44247,AHS Cancer Control Alberta,16
23712,Takeda,254
27905,Joel Neal,2


Study design extraction


,Study Design,allocation,intervention_model,masking,primary_purpose
22202,Allocation: NA | Intervention Model: SEQUENTIA...,NA,SEQUENTIAL,NONE,TREATMENT
45439,Allocation: RANDOMIZED | Intervention Model: P...,RANDOMIZED,PARALLEL,NONE,TREATMENT
44247,Allocation: NA | Intervention Model: SINGLE_GR...,NA,SINGLE_GROUP,NONE,TREATMENT
23712,Allocation: NON_RANDOMIZED | Intervention Mode...,NON_RANDOMIZED,SEQUENTIAL,NONE,TREATMENT
27905,Allocation: NA | Intervention Model: SINGLE_GR...,NA,SINGLE_GROUP,NONE,TREATMENT


Condition and intervention features


,Conditions,condition_count,Interventions,intervention_count,has_drug,has_device,has_biological,has_procedure,has_behavioral,has_diagnostic_test
22202,Sarcoma|Melanoma|Lung Cancer|Breast Cancer|Eso...,7,DRUG: PRT1419,1,1,0,0,0,0,0
45439,Type 2 Diabetes,1,DRUG: Semaglutide|DRUG: Insulin Degludec|DRUG:...,3,1,0,0,0,0,0
44247,Rectal Cancer,1,DRUG: Rosuvastatin,1,1,0,0,0,0,0
23712,Refractory Lupus Nephritis|Refractory Systemic...,2,BIOLOGICAL: TAK-007|DRUG: Chemotherapy Agents,2,1,0,1,0,0,0
27905,Metastatic Non-Squamous Non-Small Cell Lung Ca...,1,BIOLOGICAL: Pembrolizumab,1,0,0,1,0,0,0


In [8]:
#STEP 8: Droping replaced raw columns and preparing final feature groups
# Raw columns already converted into cleaner engineered features

raw_cols_to_drop = [
    "Age",
    "Phases",
    "Sponsor",
    "Locations",
    "Study Design"
]

X_train_model = X_train_fe.drop(columns=raw_cols_to_drop, errors="ignore")
X_test_model = X_test_fe.drop(columns=raw_cols_to_drop, errors="ignore")

print("X_train_model:", X_train_model.shape)
print("X_test_model:", X_test_model.shape)

print("\nRemaining columns:")
print(X_train_model.columns.tolist())

print("\nMissing values:")
print(X_train_model.isnull().sum().sort_values(ascending=False).head(5))

X_train_model: (48972, 34)
X_test_model: (12243, 34)

Remaining columns:
['Conditions', 'Interventions', 'Sex', 'Enrollment', 'Funder Type', 'start_year', 'condition_count', 'intervention_count', 'has_location', 'location_count', 'is_multisite', 'has_us_location', 'has_uk_location', 'has_china_location', 'has_canada_location', 'has_child', 'has_adult', 'has_older_adult', 'has_early_phase1', 'has_phase1', 'has_phase2', 'has_phase3', 'has_phase4', 'has_drug', 'has_device', 'has_biological', 'has_procedure', 'has_behavioral', 'has_diagnostic_test', 'sponsor_frequency', 'allocation', 'intervention_model', 'masking', 'primary_purpose']

Missing values:
Enrollment       7
Conditions       0
has_procedure    0
has_phase2       0
has_phase3       0
dtype: int64


In [9]:
#STEP 9: Final Preprocessing  - Imputation, scaling, OHE, TF-IDF for LR AND XGBoost
numeric_cols = [
    "Enrollment",
    "start_year",
    "condition_count",
    "intervention_count",
    "location_count",
    "sponsor_frequency"
]

binary_cols = [
    "has_location",
    "is_multisite",
    "has_us_location",
    "has_uk_location",
    "has_china_location",
    "has_canada_location",
    "has_child",
    "has_adult",
    "has_older_adult",
    "has_early_phase1",
    "has_phase1",
    "has_phase2",
    "has_phase3",
    "has_phase4",
    "has_drug",
    "has_device",
    "has_biological",
    "has_procedure",
    "has_behavioral",
    "has_diagnostic_test"
]

categorical_cols = [
    "Sex",
    "Funder Type",
    "allocation",
    "intervention_model",
    "masking",
    "primary_purpose"
]

# Limit the number of TF-IDF text features so Conditions and Interventions do not create thousands of columns.
max_text_features = 200


# Logistic Regression is sensitive to feature scale. Therefore, numeric columns are imputed and then scaled.
numeric_pipe_logistic = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# XGBoost is tree-based and does not need numeric scaling.Therefore, numeric columns are only imputed.
numeric_pipe_xgb = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

# Categorical columns: Missing values are filled with "Unknown", then One-Hot Encoding converts them into 0/1 columns.
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

# Preprocessor for Logistic Regression:
preprocessor_logistic = ColumnTransformer([
    ("numeric", numeric_pipe_logistic, numeric_cols),
    ("binary", "passthrough", binary_cols),
    ("categorical", categorical_pipe, categorical_cols),
    ("conditions_text", TfidfVectorizer(max_features=max_text_features), "Conditions"),
    ("interventions_text", TfidfVectorizer(max_features=max_text_features), "Interventions")
])

# Preprocessor for XGBoost:
preprocessor_xgb = ColumnTransformer([
    ("numeric", numeric_pipe_xgb, numeric_cols),
    ("binary", "passthrough", binary_cols),
    ("categorical", categorical_pipe, categorical_cols),
    ("conditions_text", TfidfVectorizer(max_features=max_text_features), "Conditions"),
    ("interventions_text", TfidfVectorizer(max_features=max_text_features), "Interventions")
])



# Fit preprocessing on training data only.
X_train_logistic = preprocessor_logistic.fit_transform(X_train_model)

# Apply the same learned preprocessing rules to the test set.
X_test_logistic = preprocessor_logistic.transform(X_test_model)

# Repeat the same process for the XGBoost-specific preprocessor.
X_train_xgb = preprocessor_xgb.fit_transform(X_train_model)
X_test_xgb = preprocessor_xgb.transform(X_test_model)

print("Logistic Regression data:", X_train_logistic.shape, X_test_logistic.shape)
print("XGBoost data:", X_train_xgb.shape, X_test_xgb.shape)

Logistic Regression data: (48972, 477) (12243, 477)
XGBoost data: (48972, 477) (12243, 477)


In [10]:
#STEP 10: Training with Logistic Regression
log_reg_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

log_reg_model.fit(X_train_logistic, y_train)

print("Logistic Regression trained.")

# Evaluate Logistic regression
y_pred = log_reg_model.predict(X_test_logistic)
y_prob_completed = log_reg_model.predict_proba(X_test_logistic)[:, 1]

print("Logistic Regression Evaluation")
print("=" * 40)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_completed))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Not Completed", "Completed"]
))

Logistic Regression trained.
Logistic Regression Evaluation
Accuracy: 0.695091072449563
ROC-AUC: 0.761323180154349

Confusion matrix:
[[1959  890]
 [2843 6551]]

Classification report:
               precision    recall  f1-score   support

Not Completed       0.41      0.69      0.51      2849
    Completed       0.88      0.70      0.78      9394

     accuracy                           0.70     12243
    macro avg       0.64      0.69      0.65     12243
 weighted avg       0.77      0.70      0.72     12243



In [11]:
#STEP 11: Training with XGboost
from xgboost import XGBClassifier

sample_weights = compute_sample_weight(
    class_weight="balanced",
    y=y_train
)

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train_xgb,
    y_train,
    sample_weight=sample_weights
)

print("XGBoost trained.")


# Evaluate XGBoost
xgb_pred = xgb_model.predict(X_test_xgb)
xgb_prob_completed = xgb_model.predict_proba(X_test_xgb)[:, 1]

print("XGBoost Evaluation")
print("=" * 40)

print("Accuracy:", accuracy_score(y_test, xgb_pred))
print("ROC-AUC:", roc_auc_score(y_test, xgb_prob_completed))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, xgb_pred))

print("\nClassification report:")
print(classification_report(
    y_test,
    xgb_pred,
    target_names=["Not Completed", "Completed"]
))

XGBoost trained.
XGBoost Evaluation
Accuracy: 0.8507718696397941
ROC-AUC: 0.8885740904050463

Confusion matrix:
[[2089  760]
 [1067 8327]]

Classification report:
               precision    recall  f1-score   support

Not Completed       0.66      0.73      0.70      2849
    Completed       0.92      0.89      0.90      9394

     accuracy                           0.85     12243
    macro avg       0.79      0.81      0.80     12243
 weighted avg       0.86      0.85      0.85     12243



In [12]:
#STEP 12: Comparison 

comparison = pd.DataFrame({
    "Model": ["Logistic Regression", "XGBoost"],

    "Accuracy": [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, xgb_pred)
    ],

    "ROC_AUC": [
        roc_auc_score(y_test, y_prob_completed),
        roc_auc_score(y_test, xgb_prob_completed)
    ],

    "Precision_Completed": [
        precision_score(y_test, y_pred, pos_label=1),
        precision_score(y_test, xgb_pred, pos_label=1)
    ],

    "Recall_Completed": [
        recall_score(y_test, y_pred, pos_label=1),
        recall_score(y_test, xgb_pred, pos_label=1)
    ],

    "F1_Completed": [
        f1_score(y_test, y_pred, pos_label=1),
        f1_score(y_test, xgb_pred, pos_label=1)
    ],

    "Precision_Not_Completed": [
        precision_score(y_test, y_pred, pos_label=0),
        precision_score(y_test, xgb_pred, pos_label=0)
    ],

    "Recall_Not_Completed": [
        recall_score(y_test, y_pred, pos_label=0),
        recall_score(y_test, xgb_pred, pos_label=0)
    ],

    "F1_Not_Completed": [
        f1_score(y_test, y_pred, pos_label=0),
        f1_score(y_test, xgb_pred, pos_label=0)
    ]
})

comparison.round(3)

,Model,Accuracy,ROC_AUC,Precision_Completed,Recall_Completed,F1_Completed,Precision_Not_Completed,Recall_Not_Completed,F1_Not_Completed
0,Logistic Regression,0.695,0.761,0.880,0.697,0.778,0.408,0.688,0.512
1,XGBoost,0.851,0.889,0.916,0.886,0.901,0.662,0.733,0.696


In [13]:
#STEP 13: Final model selection
final_model_name = "XGBoost"
final_model = xgb_model

print("Final selected model:", final_model_name)

Final selected model: XGBoost


In [14]:
# STEP 14:  Predict class and completion probability on the test set
test_pred = final_model.predict(X_test_xgb)
test_prob_completed = final_model.predict_proba(X_test_xgb)[:, 1]

# Convert probability into dashboard category
def completion_category(prob):
    if prob >= 0.55:
        return "Likely to complete"
    elif prob >= 0.45:
        return "Uncertain / borderline"
    else:
        return "High risk of non-completion"

# Add NCT Number back for identification only
test_ids = df.loc[X_test.index, "NCT Number"]

prediction_results = pd.DataFrame({
    "NCT Number": test_ids.values,
    "Actual Status": y_test.values,
    "Predicted Status": test_pred,
    "Completion Probability": test_prob_completed
})

prediction_results["Non-completion Risk"] = 1 - prediction_results["Completion Probability"]
prediction_results["Dashboard Category"] = prediction_results["Completion Probability"].apply(completion_category)

prediction_results.head(10)

,NCT Number,Actual Status,Predicted Status,Completion Probability,Non-completion Risk,Dashboard Category
0,NCT06973681,1,1,0.879824,0.120176,Likely to complete
1,NCT04172987,1,1,0.925618,0.074382,Likely to complete
2,NCT02733029,1,0,0.143766,0.856234,High risk of non-completion
3,NCT06624943,1,1,0.931535,0.068465,Likely to complete
4,NCT04112303,1,1,0.595423,0.404577,Likely to complete
5,NCT03330405,0,0,0.392022,0.607978,High risk of non-completion
6,NCT04549428,0,0,0.456908,0.543092,Uncertain / borderline
7,NCT02933060,1,1,0.694773,0.305227,Likely to complete
8,NCT05033431,1,1,0.930993,0.069007,Likely to complete
9,NCT02972554,1,1,0.826519,0.173481,Likely to complete


In [15]:
# STEP 15: SHAP Feature Importance for Final XGBoost Model

def explain_single_prediction(X_single, final_model, preprocessor_xgb, top_n=5):
    import shap
    import numpy as np
    import pandas as pd

    # Convert sparse matrix to dense if needed
    if hasattr(X_single, "toarray"):
        X_single_dense = X_single.toarray()
    else:
        X_single_dense = X_single

    # Get final model feature names
    feature_names = preprocessor_xgb.get_feature_names_out()

    # Create dataframe for SHAP
    X_single_df = pd.DataFrame(X_single_dense, columns=feature_names)

    # Predict class and probabilities
    predicted_class = final_model.predict(X_single)[0]
    probabilities = final_model.predict_proba(X_single)[0]

    if predicted_class == 1:
        predicted_outcome = "Completed"
        probability_label = "Completion probability"
        predicted_probability = probabilities[1]
    else:
        predicted_outcome = "Not Completed"
        probability_label = "Non-completion probability"
        predicted_probability = probabilities[0]

    # SHAP explainer
    explainer = shap.TreeExplainer(final_model)
    shap_values = explainer.shap_values(X_single_df)

    # If SHAP returns list, use class 1 = Completed
    if isinstance(shap_values, list):
        shap_values = shap_values[1]

    # Clean feature names
    def clean_feature_name(feature):
        feature = str(feature)

        if feature.startswith("numeric__"):
            name = feature.replace("numeric__", "")

        elif feature.startswith("binary__"):
            name = feature.replace("binary__", "")

        elif feature.startswith("categorical__"):
            name = feature.replace("categorical__", "")

        elif feature.startswith("conditions_text__"):
            term = feature.replace("conditions_text__", "").replace("_", " ")
            return "Condition text signal: " + term

        elif feature.startswith("interventions_text__"):
            term = feature.replace("interventions_text__", "").replace("_", " ")
            return "Intervention text signal: " + term

        else:
            name = feature

        name = name.replace("_", " ")

        readable_names = {
            "Enrollment": "Enrollment",
            "start year": "Start year",
            "condition count": "Condition count",
            "intervention count": "Intervention count",
            "location count": "Location count",
            "sponsor frequency": "Sponsor frequency",

            "has location": "Has location",
            "is multisite": "Multisite trial",
            "has us location": "US location",
            "has uk location": "UK location",
            "has china location": "China location",
            "has canada location": "Canada location",

            "has child": "Includes children",
            "has adult": "Includes adults",
            "has older adult": "Includes older adults",

            "has early phase1": "Early Phase 1",
            "has phase1": "Phase 1",
            "has phase2": "Phase 2",
            "has phase3": "Phase 3",
            "has phase4": "Phase 4",

            "has drug": "Drug intervention",
            "has device": "Device intervention",
            "has biological": "Biological intervention",
            "has procedure": "Procedure intervention",
            "has behavioral": "Behavioural intervention",
            "has diagnostic test": "Diagnostic test intervention",

            "Sex MALE": "Sex: Male",
            "Sex FEMALE": "Sex: Female",
            "Sex ALL": "Sex: All",

            "Funder Type INDUSTRY": "Funder type: Industry",
            "Funder Type OTHER": "Funder type: Other",
            "Funder Type NIH": "Funder type: NIH",
            "Funder Type FED": "Funder type: Federal",
            "Funder Type NETWORK": "Funder type: Network",
            "Funder Type INDIV": "Funder type: Individual",
        }

        name = readable_names.get(name, name)

        name = name.replace("allocation ", "Allocation: ")
        name = name.replace("intervention model ", "Intervention model: ")
        name = name.replace("masking ", "Masking: ")
        name = name.replace("primary purpose ", "Primary purpose: ")

        return name.strip()

    # Format actual feature value where useful
    def format_feature_value(feature, value):
        feature = str(feature)

        if value == 0:
            return ""

        if feature.startswith("numeric__"):
            clean_name = feature.replace("numeric__", "")

            if clean_name == "Enrollment":
                return f": {int(round(value))} participants"
            elif clean_name == "location_count":
                return f": {int(round(value))} locations"
            elif clean_name == "condition_count":
                return f": {int(round(value))} conditions"
            elif clean_name == "intervention_count":
                return f": {int(round(value))} interventions"
            elif clean_name == "start_year":
                return f": {int(round(value))}"
            else:
                return f": {round(value, 2)}"

        # For one-hot/categorical/binary/text features, value > 0 means present
        return ""

    # Create contribution table
    contribution_table = pd.DataFrame({
        "Feature": feature_names,
        "Feature Value": X_single_df.iloc[0].values,
        "SHAP Value": shap_values[0],
        "Contribution Strength": np.abs(shap_values[0])
    })

    # Remove features with zero value where possible, so inactive one-hot/text features do not appear
    contribution_table = contribution_table[
        contribution_table["Feature Value"] != 0
    ]

    # Add direction
    contribution_table["Direction"] = contribution_table["SHAP Value"].apply(
        lambda x: "pushed toward Completed" if x > 0 else "pushed toward Not Completed"
    )

    # Sort strongest contributors first
    contribution_table = contribution_table.sort_values(
        by="Contribution Strength",
        ascending=False
    )

    # Get top features
    top_features = contribution_table.head(top_n).copy()

    top_features["Readable Feature"] = top_features["Feature"].apply(clean_feature_name)

    top_features["Value Text"] = top_features.apply(
        lambda row: format_feature_value(row["Feature"], row["Feature Value"]),
        axis=1
    )

    # Print clean explanation
    print("Prediction Explanation")
    print("----------------------")
    print("Predicted outcome:", predicted_outcome)
    print(f"{probability_label}: {predicted_probability * 100:.1f}%")

    print("\nMain features influencing this prediction:")
    for i, row in enumerate(top_features.itertuples(), start=1):
        print(f"{i}. {row._6}{row._7} — {row.Direction}")

    return {
        "predicted_outcome": predicted_outcome,
        "probability_label": probability_label,
        "predicted_probability": predicted_probability,
        "top_features": top_features[["Readable Feature", "Value Text", "Direction"]]
    }



In [16]:
row_number = 10
X_single = X_test_xgb[row_number:row_number + 1]

result = explain_single_prediction(
    X_single=X_single,
    final_model=final_model,
    preprocessor_xgb=preprocessor_xgb,
    top_n=5
)

Prediction Explanation
----------------------
Predicted outcome: Completed
Completion probability: 91.5%

Main features influencing this prediction:
1. Enrollment: 60 participants — pushed toward Completed
2. Location count: 1 locations — pushed toward Completed
3. Start year: 2022 — pushed toward Not Completed
4. Phase 1 — pushed toward Completed
5. Includes older adults — pushed toward Not Completed


In [17]:
joblib.dump(builder, "feature_builder.pkl")
joblib.dump(raw_cols_to_drop, "raw_cols_to_drop.pkl")
joblib.dump(preprocessor_xgb, "completion_preprocessor.pkl")
joblib.dump(xgb_model, "completion_xgboost_model.pkl")

print("Saved files:")
print("feature_builder.pkl")
print("raw_cols_to_drop.pkl")
print("completion_preprocessor.pkl")
print("completion_xgboost_model.pkl")


Saved files:
feature_builder.pkl
raw_cols_to_drop.pkl
completion_preprocessor.pkl
completion_xgboost_model.pkl
